# 03 — LLM summarization & prompt strategies

**Prerequisites**
- **`notebooks/01_preprocessing.ipynb`** completed → `data/processed/monthly_statements.json`
- **vLLM** from **cbe-ai-assistant** (or FinSight `docker compose --profile infra up -d vllm-chat`): chat API on **`http://localhost:5000/v1`**
- **`.env`**: `LLM_BACKEND=vllm` and either FinSight vars (`OPENAI_COMPAT_BASE_URL`, `LLAMA_MODEL`, `MISTRAL_MODEL`) or cbe vars (`VLLM_BASE_URL`, `VLLM_CHAT_MODEL`). Model ids must match the served `--model` (default `TheBloke/Mistral-7B-Instruct-v0.2-GPTQ`).

This notebook mirrors **`scripts/test_models.py`** but keeps outputs in cells for reports: **zero-shot, few-shot, chain-of-thought** (Llama track), **Mistral-track** zero-shot, **comparative evaluation**, optional **insights** and **behavioral** analysis (uses `patterns.json` if you ran Section 2).

In [4]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from models.llama_model import LlamaModel
from models.mistral_model import MistralModel
from models.resource_monitor import ResourceMonitor

MONTHLY_PATH = ROOT / "data" / "processed" / "monthly_statements.json"
PATTERNS_PATH = ROOT / "data" / "processed" / "patterns.json"

print("ROOT:", ROOT)
print("monthly_statements.json:", MONTHLY_PATH.is_file())

ROOT: /mnt/data-disk/FinSight/FinSight-AI
monthly_statements.json: True


In [5]:
# --- Load one monthly statement -------------------------------------------
if not MONTHLY_PATH.is_file():
    raise FileNotFoundError(f"Run 01_preprocessing first. Missing {MONTHLY_PATH}")

with open(MONTHLY_PATH, encoding="utf-8") as f:
    monthly = json.load(f)

# Set to a specific "YYYY-MM" key, or None to use the first month alphabetically
STATEMENT_KEY = None

keys = sorted(monthly.keys())
key = STATEMENT_KEY if STATEMENT_KEY in monthly else keys[0]
statement = monthly[key]
print("Period:", key)
print("Keys in statement:", list(statement.keys())[:12], "...")

Period: 2024-01
Keys in statement: ['month', 'transaction_count', 'amount_total', 'total_received', 'total_spent_non_deposit', 'fraud_count', 'anomaly_count', 'by_category'] ...


In [6]:
import urllib.request

from models.chat_backend import llm_backend, openai_compat_base_url

mon = ResourceMonitor()
print("Resource snapshot:", json.dumps(mon.snapshot(), indent=2, default=str))
print("LLM_BACKEND:", llm_backend())

base_url = openai_compat_base_url()
served: list[str] = []
try:
    with urllib.request.urlopen(f"{base_url}/models", timeout=15) as resp:
        models_payload = json.loads(resp.read().decode("utf-8"))
    served = [m.get("id") for m in models_payload.get("data", []) if isinstance(m, dict)]
    print("Chat API base:", base_url)
    print("Served model ids:", served or "(none yet — is vLLM still loading?)")
except Exception as e:
    raise RuntimeError(
        f"Could not reach vLLM at {base_url}/models: {e}\n"
        "Start cbe-ai-assistant vllm-chat (host :5000) or FinSight: "
        "docker compose --profile infra up -d vllm-chat"
    ) from e

llama = LlamaModel()
mistral = MistralModel()
print("Configured LLAMA_MODEL:", llama.model_name, "@", llama.base_url)
print("Configured MISTRAL_MODEL:", mistral.model_name, "@", mistral.base_url)
print("Llama available:", llama.is_available())
print("Mistral available:", mistral.is_available())

missing = []
if not llama.is_available():
    missing.append(llama.model_name)
if not mistral.is_available():
    missing.append(mistral.model_name)
if missing:
    raise RuntimeError(
        f"Chat model(s) not listed on vLLM: {', '.join(missing)}. "
        f"Served ids: {served}. Set LLAMA_MODEL / MISTRAL_MODEL (or VLLM_CHAT_MODEL) "
        f"to match GET {base_url}/models."
    )

Resource snapshot: {
  "ram_used_gb": 21.3527,
  "ram_total_gb": 125.7245,
  "ram_percent": 17.0,
  "cpu_percent": 2.5,
  "gpu_used_mb": null,
  "gpu_total_mb": null
}
LLM_BACKEND: vllm
Chat API base: http://localhost:5000/v1
Served model ids: ['TheBloke/Mistral-7B-Instruct-v0.2-GPTQ']
Configured LLAMA_MODEL: meta-llama/Llama-3.2-3B-Instruct @ http://localhost:5003/v1
Configured MISTRAL_MODEL: TheBloke/Mistral-7B-Instruct-v0.2-GPTQ @ http://localhost:5000/v1
Llama available: False
Mistral available: True


RuntimeError: Chat model(s) not listed on vLLM: meta-llama/Llama-3.2-3B-Instruct. Served ids: ['TheBloke/Mistral-7B-Instruct-v0.2-GPTQ']. Set LLAMA_MODEL / MISTRAL_MODEL (or VLLM_CHAT_MODEL) to match GET http://localhost:5000/v1/models.

## Track A (`LlamaModel`) — `TheBloke/Mistral-7B-Instruct-v0.2-GPTQ` via vLLM — three prompt strategies

Both experiment tracks call the same vLLM deployment; names are legacy labels from the evaluation pipeline.

In [7]:
llama_results: dict = {}

for strat in ("zero_shot", "few_shot", "chain_of_thought"):
    print(f"\n=== LlamaModel.generate_summary ({strat}) ===")

    def run_llama():
        return llama.generate_summary(statement, prompt_strategy=strat)

    out, delta = mon.measure(run_llama)
    llama_results[strat] = out
    print(out["summary"])
    meta = out["metadata"]
    print(
        f"\n[inference {meta['inference_time_sec']:.2f}s | tok/s ~{meta['tokens_per_sec']:.1f} | "
        f"peak_ram {delta['peak_ram_gb']:.2f} GB | cpu_avg {delta['cpu_avg_percent']:.1f}%]"
    )


=== LlamaModel.generate_summary (zero_shot) ===


RuntimeError: OpenAI-compatible API unreachable at http://localhost:5003/v1: <urlopen error [Errno 111] Connection refused>

## Track B (`MistralModel`) — `TheBloke/Mistral-7B-Instruct-v0.2-GPTQ` via vLLM — zero-shot summary

In [8]:
print("=== MistralModel.generate_summary (zero_shot) ===")


def run_mistral():
    return mistral.generate_summary(statement, prompt_strategy="zero_shot")


mistral_zero, d_m = mon.measure(run_mistral)
print(mistral_zero["summary"])
mm = mistral_zero["metadata"]
print(
    f"\n[inference {mm['inference_time_sec']:.2f}s | peak_ram {d_m['peak_ram_gb']:.2f} GB]"
)

=== MistralModel.generate_summary (zero_shot) ===
 During the month of 2024-01, the bank account experienced a net outflow of $-45191803842.80, with a total of 500000 transactions. The majority of the activity consisted of withdrawals, which accounted for 41.087% of the flow and involved 182316 transactions, followed by transfers, which made up 33.7524% of the flow and were recorded in 40730 transactions. Deposits and bills & purchases each represented a smaller portion of the activity.

[inference 3.06s | peak_ram 21.36 GB]


## Comparative evaluation — `MistralModel` judges Track A zero-shot (`TheBloke/Mistral-7B-Instruct-v0.2-GPTQ`)

In [9]:
z = llama_results.get("zero_shot") or {}
llama_summary_text = str(z.get("summary", ""))

if not llama_summary_text:
    print("No Llama zero-shot summary — run the Llama cell above first.")
else:
    print("=== MistralModel.evaluate_summary ===")

    def run_eval():
        return mistral.evaluate_summary(statement, llama_summary_text)

    ev, d_e = mon.measure(run_eval)
    print(json.dumps({k: v for k, v in ev.items() if k != "metadata"}, indent=2, default=str))
    em = ev.get("metadata", {})
    print(
        f"\n[inference {em.get('inference_time_sec', 0):.2f}s | peak_ram {d_e['peak_ram_gb']:.2f} GB]"
    )

No Llama zero-shot summary — run the Llama cell above first.


## Optional — actionable insights (Track A / same vLLM model)

In [ ]:
_z = llama_results.get("zero_shot") or {}
_sum = str(_z.get("summary", ""))
if _sum:
    print("=== LlamaModel.generate_insights ===")

    def run_ins():
        return llama.generate_insights(statement, _sum)

    ins, d_i = mon.measure(run_ins)
    for i, line in enumerate(ins["insights"], 1):
        print(f"{i}. {line}")
    im = ins["metadata"]
    print(
        f"\n[inference {im['inference_time_sec']:.2f}s | peak_ram {d_i['peak_ram_gb']:.2f} GB]"
    )
else:
    print("Skip insights — run the Llama strategies cell first.")

## Optional — behavioral narrative (uses Section 2 patterns if present)

In [ ]:
patterns_payload: list = []
if PATTERNS_PATH.is_file():
    with open(PATTERNS_PATH, encoding="utf-8") as f:
        pdata = json.load(f)
    # Flatten to a list of dicts for the prompt (recurring + trends as dict rows)
    patterns_payload = [
        {"kind": "recurring", **x}
        for x in pdata.get("recurring_payments", [])[:20]
    ]
    patterns_payload += [
        {"kind": "trend", "category": k, **v}
        for k, v in list(pdata.get("spending_trends", {}).items())[:20]
    ]
    for line in pdata.get("behavioral_patterns", []) or []:
        patterns_payload.append({"kind": "observation", "text": line})
else:
    print(f"No {PATTERNS_PATH} — run 02_embeddings or ignore.")

if patterns_payload:
    print("=== LlamaModel.generate_behavioral_analysis ===")

    def run_beh():
        return llama.generate_behavioral_analysis(patterns_payload)

    # generate_behavioral_analysis returns str only — wrap for measure
    txt, d_b = mon.measure(run_beh)
    print(txt)
    print(f"\n[peak_ram {d_b['peak_ram_gb']:.2f} GB | duration {d_b['duration_sec']:.2f}s]")